# ECMWF Ensemble Compound-Event Forecast Processing

Reads raw ECMWF GRIB ensemble files (one per member, per variable), applies
HadUK-Grid-derived percentile thresholds, and computes binary CD/CW
compound-event forecasts for each ensemble member. Outputs are saved as
NetCDF cubes (one per run date/time), ready for probability plotting in
`04_forecast_probability_maps.ipynb`.

**Source notebook:** `ecmwf_probabilities__1_.ipynb`

**Inputs**
| Path | Description |
|---|---|
| `{ECMWF_DATA_DIR}/ecmwf_tp_{date}_uk_{member}` | 6-hourly total precipitation (m) |
| `{ECMWF_DATA_DIR}/ecmwf_mn2t6_{date}_uk_{member}` | 6-hourly minimum 2m temperature (K) |
| `data/percentile_cubes/HadUK_minT15_regridded_linear.nc` | 15th-percentile min-temp threshold (regridded to ECMWF grid) |
| `data/percentile_cubes/HadUK_precip15P_regridded_linear.nc` | 15th-percentile precipitation threshold |
| `data/percentile_cubes/HadUK_precip85P_regridded_linear.nc` | 85th-percentile precipitation threshold |

**Outputs**  
For each `run_date_time` in `EVENTS_DICT`:
- `output/forecasts/masked_HadUK_CD_final_{run_date_time}.nc`
- `output/forecasts/masked_HadUK_CW_final_{run_date_time}.nc`

**Dependencies:** `iris`, `iris_grib`, `cf_units`

**Configuration:** Edit the paths and event dictionary below.


In [ ]:
import os
import datetime
import numpy as np
import iris
import iris_grib
import cf_units
from iris.coords import DimCoord

# ── User configuration ──────────────────────────────────────────────────────
# Directory containing raw ECMWF GRIB files
ECMWF_DATA_DIR = "/path/to/ecmwf/grib/files"

# HadUK-Grid percentile threshold cubes (regridded to match ECMWF resolution)
PERCENTILE_DIR  = "data/percentile_cubes"
MINT15_PATH     = os.path.join(PERCENTILE_DIR, "HadUK_minT15_regridded_linear.nc")
PRECIP15_PATH   = os.path.join(PERCENTILE_DIR, "HadUK_precip15P_regridded_linear.nc")
PRECIP85_PATH   = os.path.join(PERCENTILE_DIR, "HadUK_precip85P_regridded_linear.nc")

# Output directory for processed NetCDF cubes
OUTPUT_DIR = "output/forecasts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ensemble configuration
N_MEMBERS  = 51
FILL_VALUE = 500   # values above this in percentile grids are treated as ocean/missing

# ECMWF forecast lead-time configuration
# 6-hourly steps; 24-hour accumulations computed from T+48 to T+264
ACCUM_PERIOD = 24
TIME_STEP    = 6
LEAD_TIMES_FOR_CALCS    = list(range(48, 270, 6))
LEAD_TIMES_FOR_VALIDITY = list(range(72, 270, 6))

# Events to process: keys are descriptive labels, values are lists of run datetimes
# (YYYYMMDDHH format). All run datetimes across all events will be processed.
EVENTS_DICT = {
    "Event_1_11Feb2021": [
        {"lead_time": "10_days", "run_date_time": "2021020100"},
        {"lead_time": "5_days",  "run_date_time": "2021020600"},
        {"lead_time": "3_days",  "run_date_time": "2021020800"},
    ],
    "Event_2_28Feb2018": [
        {"lead_time": "10_days", "run_date_time": "2018021800"},
        {"lead_time": "5_days",  "run_date_time": "2018022300"},
        {"lead_time": "3_days",  "run_date_time": "2018022500"},
    ],
    "Event_3_29Nov2010": [
        {"lead_time": "10_days", "run_date_time": "2010111900"},
        {"lead_time": "5_days",  "run_date_time": "2010112400"},
        {"lead_time": "3_days",  "run_date_time": "2010112600"},
    ],
}

print("Configuration complete.")


## 1. Load percentile threshold cubes

In [ ]:
def mask_fill_values(cube, threshold=FILL_VALUE):
    """Replace values above `threshold` (ocean fill) with NaN in place."""
    data = cube.data.astype(np.float32)
    data[data > threshold] = np.nan
    cube.data = data
    return cube

minT15P_cube   = mask_fill_values(iris.load_cube(MINT15_PATH))
precip15P_cube = mask_fill_values(iris.load_cube(PRECIP15_PATH))
precip85P_cube = mask_fill_values(iris.load_cube(PRECIP85_PATH))

for name, cube in [("minT15P",   minT15P_cube),
                   ("precip15P", precip15P_cube),
                   ("precip85P", precip85P_cube)]:
    d = cube.data
    print(f"{name}: min={np.nanmin(d):.3f}, max={np.nanmax(d):.3f}")


## 2. GRIB loading callback and processing functions

In [ ]:
def grib_ensemble_callback(cube, field, filepath):
    """
    Iris callback to attach ensemble member and forecast reference time
    coordinates when loading GRIB2 ensemble data.
    """
    if not isinstance(field, iris_grib.GribWrapper):
        return
    n_member = field.perturbationNumber
    data_date = str(field.dataDate)
    data_time = str(field.dataTime).zfill(2)
    frt = datetime.datetime.strptime(data_date + data_time[:2], '%Y%m%d%H')
    epoch = (frt - datetime.datetime(1970, 1, 1)).total_seconds()

    cube.add_aux_coord(DimCoord(n_member, standard_name='realization'))
    cube.add_aux_coord(DimCoord(
        epoch, standard_name='forecast_reference_time',
        units=cf_units.Unit('seconds since 1970-01-01 00:00:00',
                             calendar='gregorian')
    ))


def calculate_compound_events(run_date_time):
    """
    Process all ensemble members for a single forecast run date/time and
    save CD and CW binary forecast cubes to NetCDF.

    Parameters
    ----------
    run_date_time : str  Forecast initialisation datetime in YYYYMMDDHH format

    Returns
    -------
    CD_final, CW_final : iris.cube.Cube  Merged ensemble cubes (member × time × lat × lon)
    """
    run_dt_short = run_date_time[2:]
    run_dt       = datetime.datetime(
        int(run_date_time[0:4]), int(run_date_time[4:6]),
        int(run_date_time[6:8]), int(run_date_time[8:10])
    )

    lead_constraint = iris.Constraint(
        coord_values={'forecast_period': LEAD_TIMES_FOR_CALCS}
    )

    start_dt  = datetime.timedelta(hours=min(LEAD_TIMES_FOR_VALIDITY))
    step_dt   = datetime.timedelta(hours=TIME_STEP)
    window_dt = datetime.timedelta(hours=ACCUM_PERIOD)

    validity_times = []
    for i in range(len(LEAD_TIMES_FOR_VALIDITY)):
        if i == 0:
            validity_times.append(run_dt + start_dt)
        else:
            validity_times.append(validity_times[-1] + step_dt)

    CD_all_members = iris.cube.CubeList()
    CW_all_members = iris.cube.CubeList()

    for member in range(N_MEMBERS):
        print(f"  Member {member}")

        fp_precip = os.path.join(ECMWF_DATA_DIR, f"ecmwf_tp_{run_dt_short}_uk_{member}")
        fp_minT   = os.path.join(ECMWF_DATA_DIR, f"ecmwf_mn2t6_{run_dt_short}_uk_{member}")

        precip_data = iris.load_cube(fp_precip, lead_constraint,
                                     callback=grib_ensemble_callback)
        minT_data   = iris.load_cube(fp_minT,   lead_constraint,
                                     callback=grib_ensemble_callback)

        # Unit conversions
        precip_data.data = precip_data.data * 1000   # m → mm
        precip_data.units = cf_units.Unit('mm')
        minT_data.data   = minT_data.data - 273.15   # K → °C
        minT_data.units  = cf_units.Unit('C')

        # 24-hour precipitation accumulations
        accum_cubes = iris.cube.CubeList()
        for itime, vt in enumerate(validity_times):
            start_cube = precip_data.extract(iris.Constraint(time=vt - window_dt))
            end_cube   = precip_data.extract(iris.Constraint(time=vt))
            end_cube   = end_cube.copy()
            end_cube.data = end_cube.data - start_cube.data
            accum_cubes.append(end_cube)
        processed_precip = accum_cubes.merge_cube()

        # 24-hour minimum temperature
        minT_cubes = iris.cube.CubeList()
        for itime, vt in enumerate(validity_times):
            tc = iris.Constraint(
                time=lambda cell, vt=vt: (vt - window_dt) <= cell.point <= vt
            )
            period_data = minT_data.extract(tc)
            if period_data is not None:
                minT_cubes.append(period_data.collapsed('time', iris.analysis.MIN))
        processed_minT = minT_cubes.merge_cube()

        # Compound event classification
        CD_cubes = iris.cube.CubeList()
        CW_cubes = iris.cube.CubeList()

        for itime in range(len(validity_times)):
            temp_cube   = processed_minT[itime]
            precip_cube = processed_precip[itime]

            # Mask ocean fill values
            for c in [temp_cube, precip_cube]:
                d = c.data.astype(np.float32)
                d[d > FILL_VALUE] = np.nan
                c.data = d

            land_mask    = ~np.isnan(temp_cube.data)
            temp_met     = temp_cube.data   <= minT15P_cube.data
            precip_CD    = precip_cube.data <= precip15P_cube.data
            precip_CW    = precip_cube.data >= precip85P_cube.data
            compound_CD  = temp_met & precip_CD
            compound_CW  = temp_met & precip_CW

            CD_data = np.full(temp_cube.shape, np.nan)
            CW_data = np.full(temp_cube.shape, np.nan)
            CD_data[land_mask & ~compound_CD] = 0
            CD_data[compound_CD]              = 1
            CW_data[land_mask & ~compound_CW] = 0
            CW_data[compound_CW]              = 1

            CD_cubes.append(temp_cube.copy(data=CD_data))
            CW_cubes.append(temp_cube.copy(data=CW_data))

        CD_member = CD_cubes.merge_cube()
        CW_member = CW_cubes.merge_cube()

        for cube, val in [(CD_member, member), (CW_member, member)]:
            cube.add_aux_coord(iris.coords.AuxCoord(val, long_name='ensemble_member'))

        CD_all_members.append(CD_member)
        CW_all_members.append(CW_member)

    CD_final = CD_all_members.merge_cube()
    CW_final = CW_all_members.merge_cube()

    # Save to NetCDF
    for cube, label in [(CD_final, 'CD'), (CW_final, 'CW')]:
        out_path = os.path.join(OUTPUT_DIR, f"masked_HadUK_{label}_final_{run_date_time}.nc")
        attrs_to_remove = [k for k in cube.attributes if 'GRIB' in k]
        for k in attrs_to_remove:
            del cube.attributes[k]
        iris.save(cube, out_path)
        print(f"  Saved {out_path}")

    return CD_final, CW_final


## 3. Run processing for all events

In [ ]:
for event, entries in EVENTS_DICT.items():
    print(f"\n=== {event} ===")
    for entry in entries:
        print(f"  Lead time: {entry['lead_time']}  Run: {entry['run_date_time']}")
        calculate_compound_events(entry['run_date_time'])
print("\nAll events processed.")
